In [ ]:
from datasets import load_dataset
import pandas as pd
import re

### convert FLUTE data into proper structure ###

# load data
df = pd.concat([s.to_pandas() for s in load_dataset("ColumbiaNLP/FLUTE").values()])

# remove non-idiom records
d = df[df['type'] == 'Idiom']

# filter to entailment rows, which will produce the "correct" side for each idiom. filter out irrelevant columns and rename remaining columns for clarity.
ent = d[d.label == 'Entailment'][['idiom','hypothesis','premise','explanation']].rename(
    columns={'hypothesis':'example','premise':'correct_substitution','explanation':'explanation_correct'})

# filter to contradiction rows, which will produce the "incorrect" side for each idiom
con = d[d.label == 'Contradiction'][['idiom','hypothesis','premise','explanation']].rename(
    columns={'hypothesis':'example','premise':'incorrect_substitution','explanation':'explanation_incorrect'})

# join transposed sides together
wide = ent.merge(con, on=['idiom','example'])
wide.to_csv('flute_idioms.csv', index=False)


### prep MAGPIE data ###

# load data
URL = "https://huggingface.co/api/datasets/gsarti/magpie/parquet/magpie/train/0.parquet"
magpie = pd.read_parquet(URL)

# remove literal records
m = magpie[magpie['usage'] == 'figurative'].copy()

# filter to idiom and sentence columns since those are the only ones that match FLUTE
m = m[['idiom', 'sentence']].rename(columns={'sentence': 'example'})

# list idioms from FLUTE
flute_set = set(wide['idiom'].str.lower().str.strip())

# remove FLUTE idioms from MAGPIE corpus
m = m[~m['idiom'].str.lower().str.strip().isin(flute_set)]

# add blank columns for proper data structure
for c in ['correct_substitution','explanation_correct','incorrect_substitution','explanation_incorrect']:
    m[c] = ''

# the BNC text that MAGPIE was built from is pre-tokenized, so punctuation is spaced out
# close it up so the sentences read normally when writing pairs by hand
def clean(t):
    t = str(t)
    t = re.sub(r"\s+([,.;:!?’”])", r"\1", t)   # "word , word" -> "word, word"
    t = re.sub(r"([‘“])\s+", r"\1", t)          # "‘ word" -> "‘word"
    t = re.sub(r"\s+-\s+", "-", t)              # "hair's - breadth" -> "hair's-breadth"
    return re.sub(r"\s+", " ", t).strip()

m['example'] = m['example'].astype(str).map(clean)
s = m['example']
wc = s.str.split().str.len()

# hard filters: any one of these disqualifies a sentence outright
bad = (
    # offensive content (also covers the screening promised in the proposal)
    s.str.contains(r'\b(?:fuck\w*|shit\w*|cunt\w*|bastard|bollocks|wank\w*|piss\w*|tits|arse\w*|nigg\w*|paki)\b', case=False)
    # spoken-transcript filler
  | s.str.contains(r'\b(?:er|erm|mm|mmm|hmm|innit|dunno)\b', case=False)
    # stutter repeats from transcription: "it's it's all a bit..."
  | s.str.contains(r'\b(\w+) \1\b', case=False)
    # too short to write a premise against, or too long to bother matching
  | (wc < 8) | (wc > 22)
    # fragments with no sentence-ending punctuation
  | ~s.str.strip().str.endswith(('.','!','?'))
    # financial or statistical prose
  | s.str.contains(r'[£$€%]')
    # 2+ mid-sentence capitalised words = proper-noun heavy, needs outside context
  | (s.str.count(r'(?<=[a-z,] )[A-Z][a-z]{2,}') >= 2)
)
m, s, wc = m[~bad], s[~bad], wc[~bad]

# soft score out of 6, used to rank whatever survived
m = m.assign(score =
      wc.between(10, 18).astype(int) * 2                # comfortable length
    + (~s.str.contains(r'[‘’“”"]')).astype(int) * 2     # not embedded dialogue
    + (~s.str.contains(r'\d')).astype(int)              # no figures
    + (s.str.count(r',') <= 1).astype(int)              # simple clause structure
)

# best sentence per idiom, then sample 100 from the top-scoring pool
best = m.sort_values('score', ascending=False).groupby('idiom', as_index=False).first()
#selected = best.nlargest(400, 'score').sample(100, random_state=42).sort_values('idiom') if we decide we want to sample
selected = best.sort_values('idiom')

selected = selected[['idiom','example','score','correct_substitution','explanation_correct',
                     'incorrect_substitution','explanation_incorrect']]
selected.to_csv('magpie_selected.csv', index=False)
print(len(selected), "idioms selected")

/tmp/ipykernel_1450/2321480769.py:68: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | s.str.contains(r'\b(\w+) \1\b', case=False)


1074 idioms selected
